# PREPROCESSING - ChurnShield AI — Explainable Customer Retention & Revenue Intelligence Platform

## 1) IMPORT LIBRARIES

In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

## 2) LOAD CLEANED DATASET

In [2]:
df = pd.read_csv("telco_cleaned.csv")

In [3]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.tail()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7038,Male,No,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,No,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,No,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,Yes,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes
7042,Male,No,No,No,66,Yes,No,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.50,No


In [5]:
df.shape

(7043, 20)

## DATA VALIDATION

In [6]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(22)

### 22 duplicate feature rows ഉണ്ടെന്നതുകൊണ്ട് മാത്രം അവ delete ചെയ്യുന്നില്ല. differnt customers-ന് ഒരേ feature values ഉണ്ടാകാം.

## 4) SIMPLIFY REDUNDANT CATEGORIES

In [8]:
addon_cols = ["OnlineSecurity","OnlineBackup", "DeviceProtection","TechSupport","StreamingTV","StreamingMovies","MultipleLines"]

In [9]:
addon_cols

['OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'MultipleLines']

In [10]:
df[addon_cols] = df[addon_cols].replace({"No internet service": "No","No phone service": "No"})

In [11]:
df[addon_cols].head()

,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,MultipleLines
0,No,Yes,No,No,No,No,No
1,Yes,No,Yes,No,No,No,No
2,Yes,Yes,No,No,No,No,No
3,Yes,No,Yes,Yes,No,No,No
4,No,No,No,No,No,No,No


## 5) TRAGET ENCODING

In [12]:
df["ChurnFlag"] = df["Churn"].map({"No": 0,"Yes": 1})

In [13]:
df["ChurnFlag"].value_counts()

ChurnFlag
0    5174
1    1869
Name: count, dtype: int64

In [14]:
print("\nTarget Distribution (%):")
print((df["ChurnFlag"].value_counts(normalize=True) * 100).round(2))


Target Distribution (%):
ChurnFlag
0    73.46
1    26.54
Name: proportion, dtype: float64


## 6) FEATURE ENGINEERING

In [15]:
service_cols = ["OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]

In [16]:
service_cols

['OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies']

##### Total number of additional services

In [17]:
df["num_addons"] = (df[service_cols] == "Yes").sum(axis=1)

##### Identify new customers

In [18]:
df["is_new_customer"] = (df["tenure"] <= 12).astype(int)

##### Average monthly spending

In [19]:
df["avg_monthly_spend"] = (df["TotalCharges"] /df["tenure"].replace(0, np.nan)).fillna(df["MonthlyCharges"])

df[["num_addons", "is_new_customer", "avg_monthly_spend"]].head()

,num_addons,is_new_customer,avg_monthly_spend
0,1,1,29.850000
1,2,0,55.573529
2,2,1,54.075000
3,3,0,40.905556
4,0,1,75.825000


## 7) REMOVE UNNECESSARY COLUMNS

In [20]:
df_model = df.drop(columns=["Churn"])
df_model.shape

(7043, 23)

#### churnFlag target നിലനിർത്തുന്നു.

## 8) SEPARATE FEATURES AND TARGET

In [21]:
X = df_model.drop(columns=["ChurnFlag"])
y = df_model["ChurnFlag"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (7043, 22)
Target Shape: (7043,)


## 9) Train-Test Split

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

y_train.name = "ChurnFlag"
y_test.name = "ChurnFlag"

In [23]:
X_train.shape

(5634, 22)

In [24]:
X_test.shape

(1409, 22)

In [25]:
print("\nTraining Target Distribution:")
print(y_train.value_counts())


Training Target Distribution:
ChurnFlag
0    4139
1    1495
Name: count, dtype: int64


## 10) Identify Column Groups

In [26]:
num_cols = ["tenure","MonthlyCharges","TotalCharges","num_addons","avg_monthly_spend"]

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

passthrough_cols = ["is_new_customer"]


In [27]:
print("Numerical Columns:", num_cols)
print("\nCategorical Columns:", cat_cols)
print("\nPassthrough Columns:", passthrough_cols)

Numerical Columns: ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_addons', 'avg_monthly_spend']

Categorical Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Passthrough Columns: ['is_new_customer']


# 11) Create Preprocessing Pipeline

In [29]:
try:
    encoder = OneHotEncoder(drop="if_binary",handle_unknown="ignore",sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(drop="if_binary",handle_unknown="ignore",sparse=False)
    
preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), num_cols),("cat", encoder, cat_cols),("pass", "passthrough", passthrough_cols)],verbose_feature_names_out=False)


In [30]:
encoder

,categories,'auto'
,drop,'if_binary'
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [31]:
preprocessor

,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


## 12) Fit and Transform

In [32]:
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_p = pd.DataFrame(X_train_p,columns=feature_names,index=X_train.index)
X_test_p = pd.DataFrame(X_test_p,columns=feature_names,index=X_test.index)

print("Training Shape:", X_train_p.shape)
print("Testing Shape:", X_test_p.shape)

Training Shape: (5634, 29)
Testing Shape: (1409, 29)


In [33]:
print("Missing Values in Train:",
      X_train_p.isnull().sum().sum())

print("Missing Values in Test:",
      X_test_p.isnull().sum().sum())

print("\nDuplicate Rows in Train:",
      X_train_p.duplicated().sum())

print("\nFeature Count:", len(feature_names))

print("\nScaled Numerical Features:")
print(X_train_p[num_cols].describe().loc[["mean", "std"]].round(2))

Missing Values in Train: 0
Missing Values in Test: 0

Duplicate Rows in Train: 26

Feature Count: 29

Scaled Numerical Features:
      tenure  MonthlyCharges  TotalCharges  num_addons  avg_monthly_spend
mean    -0.0            -0.0           0.0         0.0               -0.0
std      1.0             1.0           1.0         1.0                1.0
